# M&A Due Diligence RL Training — 3-Phase Pipeline

| Phase | Purpose | ~Time on T4 |
|-------|---------|-------------|
| **1. SFT Warm-up** | Teach JSON output format | ~5 min |
| **2. Lightweight GRPO** | Test full RL loop (10 steps) | ~10 min |
| **3. Full GRPO** | Actual training → curves | ~45-90 min |

> GPU: T4 (15 GB VRAM). Peak usage ~10 GB. `load_in_4bit=True` keeps it safe.

In [ ]:
# Cell 1: Install
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q trl datasets httpx fastapi uvicorn slowapi matplotlib pydantic
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch runtime!')

In [ ]:
# Cell 2: Setup — clone repo and find project root
import os, sys, glob

# Uncomment ONE of these:
# Option A: clone from GitHub
!git clone https://github.com/NJVinay/openenv_ma_analyzer.git

# Option B: upload zip via Colab file panel, then:
# import zipfile; zipfile.ZipFile('output.zip').extractall('.')

# Auto-detect project root: find the directory that contains server/
def find_project_root(start='.', max_depth=4):
    # Check current dir first
    if os.path.isdir(os.path.join(start, 'server')):
        return os.path.abspath(start)
    # Search subdirs
    for depth in range(1, max_depth + 1):
        pattern = os.path.join(start, *(['*'] * depth), 'server')
        for match in glob.glob(pattern):
            if os.path.isdir(match):
                return os.path.abspath(os.path.dirname(match))
    return os.path.abspath(start)

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Verify critical dirs exist
for d in ['server', 'security', 'training']:
    assert os.path.isdir(d), f'Missing {d}/ in {PROJECT_ROOT}'

print(f'Project root: {PROJECT_ROOT}')
print(f'Files: {sorted(os.listdir("."))}')

In [ ]:
# Cell 3: Start environment server
import subprocess, time, httpx, sys, tempfile, os

# Capture stderr so server import errors are visible
_server_log = os.path.join(tempfile.gettempdir(), 'ma_server.log')
_server_logf = open(_server_log, 'w')

# CRITICAL: pass cwd + PYTHONPATH so the subprocess can find server/
_env = {**os.environ, 'PYTHONPATH': PROJECT_ROOT}
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '7860'],
    stdout=_server_logf, stderr=_server_logf,
    cwd=PROJECT_ROOT,
    env=_env,
)
time.sleep(5)

try:
    r = httpx.get('http://localhost:7860/health', timeout=10)
    assert r.status_code == 200
    print('Server healthy:', r.json())
except Exception as e:
    _server_logf.flush()
    print(f'Server failed to start: {e}')
    print('--- Server log (last 50 lines) ---')
    with open(_server_log) as f:
        lines = f.readlines()
        for line in lines[-50:]:
            print(line, end='')
    print('--- End server log ---')
    raise

In [ ]:
# Cell 4: Load Qwen2.5-3B with Unsloth + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,   # ~2.5 GB VRAM on T4
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model on device:', next(model.parameters()).device)
print('LoRA applied successfully')

---
## Phase 1: SFT Warm-up
Teaches JSON format on 20 curated M&A examples.

In [ ]:
# Cell 5: SFT warm-up (T4-safe: batch=1, SFTConfig fallback)
# SFTConfig requires trl>=0.8.0 — use TrainingArguments as safe fallback
try:
    from trl import SFTTrainer, SFTConfig as _SFTArgs
    _sft_extra = dict(max_seq_length=2048, dataset_text_field='text')
    _sft_trainer_extra = {}
    print('Using SFTConfig')
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments as _SFTArgs
    _sft_extra = {}
    _sft_trainer_extra = dict(max_seq_length=2048, dataset_text_field='text')
    print('Using TrainingArguments fallback')

from datasets import Dataset
from training.sft_data import SFT_EXAMPLES

MA_SYSTEM_PROMPT = (
    'You are an expert M&A due diligence analyst.\n'
    'You review NDAs, Letters of Intent, Share Purchase Agreements,\n'
    'and Representations & Warranties to identify risk clauses,\n'
    'quantify deal exposure, and rewrite unfavourable terms.\n'
    'Always respond in the JSON format specified in the task prompt.\n'
    'Reason from the document content - not from memorised templates.'
)

def format_prompt(task_prompt, deal_excerpt):
    return (f'<|system|>\n{MA_SYSTEM_PROMPT}\n'
            f'<|user|>\n{task_prompt}\n\nDocument:\n{deal_excerpt}\n'
            f'<|assistant|>\n')

def fmt_sft(ex):
    return {'text': f'<|system|>\n{MA_SYSTEM_PROMPT}\n<|user|>\n{ex["prompt"]}\n<|assistant|>\n{ex["completion"]}'}

sft_dataset = Dataset.from_list(SFT_EXAMPLES).map(fmt_sft)

sft_args = _SFTArgs(
    output_dir='./sft-checkpoint',
    num_train_epochs=8,  # Increased from 3 to 8 for better formatting compliance
    per_device_train_batch_size=1,  # T4 safe
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=999,
    report_to='none',
    **_sft_extra,
)

sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_dataset, args=sft_args,
    **_sft_trainer_extra,
)
print(f'SFT: {len(sft_dataset)} examples x3 epochs')
sft_trainer.train()
sft_log = sft_trainer.state.log_history
print('Phase 1 complete!')

In [ ]:
# Cell 6: Verify SFT — check JSON compliance
import json, torch
from client import MADueDiligenceClient
from models import Action

env = MADueDiligenceClient(base_url='http://localhost:7860')
FastLanguageModel.for_inference(model)

json_ok = 0
for i in range(5):
    obs = env.reset(seed=i)
    inputs = tokenizer(format_prompt(obs.task_prompt, obs.deal_excerpt),
                       return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256,
                             do_sample=True, temperature=0.7)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()
    try:
        # Robust JSON parsing: strip markdown code blocks if present
        clean_text = text
        if '```json' in clean_text:
            clean_text = clean_text.split('```json')[1]
        if '```' in clean_text:
            clean_text = clean_text.split('```')[0]
        clean_text = clean_text.strip()
        
        parsed = json.loads(clean_text)
        json_ok += 1
        print(f'[{i}] OK keys={list(parsed.keys())}')
    except json.JSONDecodeError:
        print(f'[{i}] BAD: {text[:100]}...')

print(f'JSON compliance: {json_ok}/5')
assert json_ok >= 2, 'Too few valid JSON outputs — re-run SFT with more epochs'
FastLanguageModel.for_training(model)

---
## Phase 2: Lightweight GRPO (10-step bug catcher)
Validates the full loop before committing to a long run.

In [ ]:
# Cell 7: Reward function
import json, traceback

def ma_reward_func(completions, **kwargs):
    """Called by GRPOTrainer. Sends each completion to the M&A env."""
    rewards = []
    for completion in completions:
        try:
            text = completion if isinstance(completion, str) else str(completion)
            if '<|assistant|>' in text:
                text = text.split('<|assistant|>')[-1].strip()
            # Auto-detect action type from JSON keys
            action_type = 'identify'
            try:
                # Robust JSON parsing
                clean_text = text
                if '```json' in clean_text:
                    clean_text = clean_text.split('```json')[1]
                if '```' in clean_text:
                    clean_text = clean_text.split('```')[0]
                clean_text = clean_text.strip()
                
                p = json.loads(clean_text)
                if 'risk_level' in p: action_type = 'assess'
                elif 'rewritten_clause' in p: action_type = 'rewrite'
            except Exception: pass
            obs = env.reset()
            obs2 = env.step(Action(agent_output=text, action_type=action_type))
            rewards.append(float(obs2.reward or 0.0))
        except Exception as e:
            print(f'Reward error: {e}')
            rewards.append(0.0)
    return rewards

print('Reward function ready')

In [ ]:
# Cell 8: Phase 2 — 10-step GRPO test
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import inspect

obs0 = env.reset(seed=0)
test_dataset = Dataset.from_dict(
    {'prompt': [format_prompt(obs0.task_prompt, obs0.deal_excerpt)] * 10}
)

test_cfg = GRPOConfig(
    output_dir='./grpo-test',
    max_steps=10,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=256,
    logging_steps=1,
    report_to='none',
)

# TRL API varies across versions — detect the correct kwarg name
_grpo_params = inspect.signature(GRPOTrainer.__init__).parameters
_reward_kwargs = {}
if 'reward_funcs' in _grpo_params:
    _reward_kwargs['reward_funcs'] = [ma_reward_func]
    print('TRL API: using reward_funcs (list)')
elif 'reward_func' in _grpo_params:
    _reward_kwargs['reward_func'] = ma_reward_func
    print('TRL API: using reward_func (singular)')
elif 'reward_fn' in _grpo_params:
    _reward_kwargs['reward_fn'] = ma_reward_func
    print('TRL API: using reward_fn')
else:
    # Fallback: try the most common
    _reward_kwargs['reward_funcs'] = [ma_reward_func]
    print('TRL API: defaulting to reward_funcs (list)')

test_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=test_dataset, args=test_cfg,
    **_reward_kwargs,
)
print('Phase 2: 10-step test...')
try:
    test_trainer.train()
    print('Phase 2 PASSED')
    phase2_ok = True
except Exception as e:
    traceback.print_exc()
    phase2_ok = False
    print('Phase 2 FAILED — fix before Phase 3!')

---
## Phase 3: Full GRPO Training
Only runs if Phase 2 passed. Produces reward + loss curves.

In [ ]:
# Cell 9: Full GRPO
assert phase2_ok, 'Phase 2 failed — fix bugs first!'

# Build diverse prompts bypassing the HTTP rate limiter (slowapi) by using the python class directly
from server.environment import MADueDiligenceEnvironment
local_env = MADueDiligenceEnvironment()
full_prompts = []
for i in range(200):
    obs = local_env.reset(seed=i)
    full_prompts.append(format_prompt(obs.task_prompt, obs.deal_excerpt))

full_dataset = Dataset.from_dict({'prompt': full_prompts})

# T4 note: grad_accum=8 (~45 min). Use 64 on A100 for better estimates.
full_cfg = GRPOConfig(
    output_dir='./ma-rl-checkpoints',
    num_train_epochs=1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_completion_length=256,
    logging_steps=5,
    save_steps=25,
    report_to='none',
)
# Reuse the same reward kwarg detected in Phase 2
full_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=full_dataset, args=full_cfg,
    **_reward_kwargs,
)
print('Phase 3: Full GRPO starting...')
full_trainer.train()
print('Phase 3 complete!')

In [ ]:
# Cell 10: Extract metrics (handles all TRL key variants)
log = full_trainer.state.log_history
print('All logged keys:', set(k for x in log for k in x))

def extract(entries, keys):
    for k in keys:
        s = [x['step'] for x in entries if k in x]
        v = [x[k] for x in entries if k in x]
        if s:
            print(f'  key={k!r} ({len(s)} pts)')
            return s, v
    print(f'  Not found among: {keys}')
    return [], []

reward_steps, reward_vals = extract(log,
    ['reward', 'rewards/mean', 'reward_mean', 'train/reward', 'rewards'])
loss_steps, loss_vals = extract(log,
    ['loss', 'train/loss', 'train_loss'])
print(f'Rewards: {len(reward_steps)} pts | Loss: {len(loss_steps)} pts')

In [ ]:
# Cell 11: reward_curve.png
import matplotlib.pyplot as plt, numpy as np

fig, ax = plt.subplots(figsize=(10, 6))
if reward_steps:
    ax.plot(reward_steps, reward_vals, '#2563eb', lw=2, label='GRPO reward')
    if len(reward_steps) > 3:
        z = np.polyfit(reward_steps, reward_vals, 1)
        ax.plot(reward_steps, np.poly1d(z)(reward_steps),
                '--', color='#93c5fd', alpha=0.7, label='trend')
    ax.legend()
    ax.set_ylabel('Mean Reward')
else:
    # Fallback to SFT loss
    ss = [x.get('step', i) for i, x in enumerate(sft_log) if 'loss' in x]
    sv = [x['loss'] for x in sft_log if 'loss' in x]
    ax.plot(ss, sv, '#2563eb', lw=2)
    ax.set_ylabel('SFT Loss (GRPO rewards not logged)')

ax.set_xlabel('Training Step')
ax.set_title('M&A RL - Reward Curve (GRPO, Qwen2.5-3B)')
ax.grid(True, alpha=0.3)
fig.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved reward_curve.png')

In [ ]:
# Cell 12: loss_curve.png (SFT + GRPO combined)
import matplotlib.pyplot as plt

fig2, ax2 = plt.subplots(figsize=(10, 6))
ss = [x.get('step', i) for i, x in enumerate(sft_log) if 'loss' in x]
sv = [x['loss'] for x in sft_log if 'loss' in x]
if ss:
    ax2.plot(ss, sv, '#f59e0b', lw=2, label='Phase 1: SFT')
if loss_steps:
    offset = (max(ss) + 1) if ss else 0
    ax2.plot([s + offset for s in loss_steps], loss_vals,
             '#dc2626', lw=2, label='Phase 3: GRPO')
if not ss and not loss_steps:
    ax2.text(0.5, 0.5, 'No loss data', transform=ax2.transAxes, ha='center')

ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss')
ax2.set_title('M&A RL - Loss Curve (SFT + GRPO, Qwen2.5-3B)')
ax2.grid(True, alpha=0.3)
if ss or loss_steps: ax2.legend()
fig2.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved loss_curve.png')

In [ ]:
# Cell 13: Post-training output samples (hacking check)
FastLanguageModel.for_inference(model)
print('=== Post-Training Samples ===')
for i in range(5):
    obs = env.reset(seed=100+i)
    inputs = tokenizer(format_prompt(obs.task_prompt, obs.deal_excerpt),
                       return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256,
                             do_sample=True, temperature=0.7)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()
    obs2 = env.step(Action(agent_output=text, action_type='identify'))
    print(f'[{i+1}] reward={obs2.reward:.3f} | {text[:180]}')

In [ ]:
# Cell 14: Save — MUST use save_pretrained_merged (not save_pretrained)
FastLanguageModel.for_training(model)
model.save_pretrained_merged('final_model', tokenizer, save_method='merged_16bit')
print('Saved: final_model/')

In [ ]:
# Cell 15: Commit training curves to repo
!git add reward_curve.png loss_curve.png
!git commit -m 'Add training curves (SFT+GRPO, Qwen2.5-3B, T4)'
!git push

In [ ]:
# Cell 16: Cleanup
server_proc.terminate()
print('Done! Artifacts: reward_curve.png  loss_curve.png  final_model/')